<a href="https://colab.research.google.com/github/minato1204love-TTP/Khang_INFO4670_Fall2026/blob/main/Week5_INFO4670_Assignment2_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [2]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [23]:
# TODO: set n_missing_study to the number of blank study_hours_reported values
# Count missing study-hours values
n_missing_study = students["study_hours_reported"].isna().sum()

print("Missing study hours:", n_missing_study)
print("Present study hours:", students["study_hours_reported"].notna().sum())

Missing study hours: 255
Present study hours: 1772


**Your justification (1–2 sentences):** _..._

In [24]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


In [25]:
students["study_hours_reported"] = students["study_hours_reported"].fillna(
    students["study_hours_reported"].median()
)

### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [26]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups
# Show categories before cleaning
print("Before:")
print(students["housing"].value_counts(dropna=False))

# Normalize text first
h = students["housing"].str.strip().str.lower()

# Standardize common variants into the three true groups
def clean_housing(x):
    if pd.isna(x):
        return x

    # On-campus variants
    if "campus" in x or "dorm" in x or "residence" in x:
        if "off" not in x:
            return "On-campus"

    # Off-campus variants
    if "off" in x or "apartment" in x:
        return "Off-campus"

    # Family/home variants
    if "family" in x or "parent" in x or x == "home":
        return "With family"

    return x

students["housing_clean"] = h.apply(clean_housing)

print("\nAfter:")
print(students["housing_clean"].value_counts(dropna=False))

Before:
housing
Off-Campus     755
On-Campus      480
With Family    420
off campus     113
Off-campus      77
with family     72
on-campus       69
 On-Campus      41
Name: count, dtype: int64

After:
housing_clean
Off-campus     945
On-campus      590
With family    492
Name: count, dtype: int64


In [27]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off-campus': 945, 'On-campus': 590, 'With family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [28]:
# TODO
# Impossible ages: negative or unrealistically above human age
impossible_ages = sorted(
    students.loc[
        (students["age"] < 0) | (students["age"] > 120),
        "age"
    ].unique().tolist()
)

# Count impossible negative work hours
n_neg_work = (students["work_hours_per_week"] < 0).sum()

print("Impossible ages:", impossible_ages)
print("Negative work-hour rows:", n_neg_work)

Impossible ages: [-22, 199, 220]
Negative work-hour rows: 4


In [29]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


In [30]:
students.loc[
    (students["age"] < 0) | (students["age"] > 120),
    "age"
] = np.nan

students.loc[
    students["work_hours_per_week"] < 0,
    "work_hours_per_week"
] = np.nan

### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [31]:
# TODO: build students_dedup (one row per student)
# Remove duplicate student IDs so the roster has one row per student
students_dedup = students.drop_duplicates(
    subset="student_id",
    keep="first"
).copy()

print("Before:", len(students))
print("After:", len(students_dedup))
print("Student IDs unique:", students_dedup["student_id"].is_unique)

Before: 2027
After: 2000
Student IDs unique: True


**Why not de-dupe enroll / activity? (1 sentence):** _..._

In [32]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [40]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
# Standardize student key: NU-#### -> integer
students_dedup["sid"] = (
    students_dedup["student_id"]
    .astype(str)
    .str.replace("NU-", "", regex=False)
    .astype(int)
)

# Inspect activity columns so we can identify its ID column
print("Activity columns:", activity.columns.tolist())

Activity columns: ['student_id', 'week', 'lms_clicks', 'minutes_active']


In [41]:
# Find activity's student-ID column
possible_id_cols = [
    c for c in activity.columns
    if c.lower() in ["sid", "student_id", "studentid"]
]

print("Possible activity ID columns:", possible_id_cols)

Possible activity ID columns: ['student_id']


In [43]:
# Standardize activity student ID
activity["sid"] = (
    activity["student_id"]
    .astype(str)
    .str.replace("NU-", "", regex=False)
    .astype(int)
)

# Summarize weekly activity to one row per student
activity_summary = (
    activity.groupby("sid", as_index=False)
    .agg(
        total_minutes_active=("minutes_active", "sum")
    )
)

# Merge activity summary with the cleaned student roster
analysis = students_dedup.merge(
    activity_summary,
    on="sid",
    how="left"
)

# Verify
print("Analysis rows:", len(analysis))
print("Unique student IDs:", analysis["student_id"].nunique())
print("Student ID unique:", analysis["student_id"].is_unique)

Analysis rows: 2000
Unique student IDs: 2000
Student ID unique: True


In [44]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [45]:
# TODO
# Standardize enrollment key
enroll["sid"] = (
    enroll["sid"]
    .astype(str)
    .str.replace("NU-", "", regex=False)
    .astype(int)
)

student_keys = set(students_dedup["sid"])

# Number of enrollment ROWS belonging to roster students
matched = enroll["sid"].isin(student_keys).sum()

# Unique orphan IDs
orphan_ids = set(enroll["sid"]) - student_keys

print("Enrollment rows:", len(enroll))
print("Matched rows:", matched)
print("Unmatched rows:", len(enroll) - matched)
print("Orphan IDs:", len(orphan_ids))
print("Final roster:", len(analysis))

Enrollment rows: 8088
Matched rows: 8041
Unmatched rows: 47
Orphan IDs: 14
Final roster: 2000


In [46]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [72]:
print(enroll.columns.tolist())

['sid', 'course', 'term', 'grade']


In [71]:
print("Total rows:", len(enroll))
print("Original blanks:", enroll["enrollment_date"].isna().sum())
print("Parsed blanks:", dates_parsed.isna().sum())

print("\nVALUES THAT FAILED:")
print(enroll.loc[dates_parsed.isna(), "enrollment_date"].value_counts(dropna=False))

Total rows: 8088


KeyError: 'enrollment_date'

In [70]:
# TODO
# Parse each date individually to handle mixed formats
dates_parsed = enroll["enrollment_date"].apply(
    lambda x: pd.to_datetime(x, errors="coerce")
)

print("Missing dates:", dates_parsed.isna().sum())

KeyError: 'enrollment_date'

In [60]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

❌ C7 not yet — parse every format so no valid date becomes NaT


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [61]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]
# Z-score study hours
analysis["study_z"] = (
    analysis["study_hours_reported"]
    - analysis["study_hours_reported"].mean()
) / analysis["study_hours_reported"].std()

# Divide GPA into four bands
analysis["gpa_band"] = pd.cut(
    analysis["final_gpa"],
    bins=[-0.01, 1, 2, 3, 4],
    labels=["0–1", "1–2", "2–3", "3–4"]
)

print("study_z mean:", analysis["study_z"].mean())

print("\nGPA bands:")
print(analysis["gpa_band"].value_counts().sort_index())

study_z mean: -5.098144129078718e-16

GPA bands:
gpa_band
0–1      70
1–2     528
2–3    1102
3–4     300
Name: count, dtype: int64


In [62]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [63]:
# TODO: write analysis to northgate_clean.csv
analysis.to_csv(
    "northgate_clean.csv",
    index=False
)

print("northgate_clean.csv saved successfully.")

northgate_clean.csv saved successfully.


In [64]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- _decision → justification_
- Missing values → I used the median for missing study hours because it is less affected by extreme values.
- Housing → I standardized the housing values into three consistent groups.
- Impossible values → I treated impossible ages and negative work hours as invalid but kept extreme values that were still possible.
- Duplicates → I removed duplicate student records so each student appears only once.
- Student key → I standardized student IDs so the three datasets could be matched correctly.
- Dates → I converted enrollment dates into one consistent date format so no valid dates were lost.
-

### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [65]:
# TODO: compute the mean GPA and explore one relationship on the CLEAN data
# Mean GPA
mean_gpa = analysis["final_gpa"].mean()

# Relationship between study hours and GPA
study_gpa_corr = analysis[
    ["study_hours_reported", "final_gpa"]
].corr().iloc[0, 1]

print("Mean GPA:", round(mean_gpa, 2))
print(
    "Correlation between study hours and GPA:",
    round(study_gpa_corr, 3)
)

Mean GPA: 2.31
Correlation between study hours and GPA: 0.62


In [66]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
